<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Notebook Introduction — Author Feedback Integration</h3>

<p>
این Notebook برای مرحله ادغام بازخورد Expert Review روی
<code>Golden Dataset v1</code>
ساخته شده است.
</p>

<p>
هدف این Notebook ایجاد یک مسیر کنترل‌شده، قابل‌ردیابی و Versioned برای بررسی بازخورد مؤلف، ثبت تصمیم‌های اصلاحی و آماده‌سازی تولید نسخه بعدی Golden Dataset است.
</p>

<p>
این مرحله بعد از تولید اولیه Golden Dataset v1 قرار دارد و قبل از ساخت Golden Dataset v2 اجرا می‌شود.
</p>

<h4>هدف اصلی Notebook</h4>

<ul>
<li>بررسی Artifactهای مربوط به Golden Dataset v1</li>
<li>بررسی Review Package ارسالشده برای Expert Review</li>
<li>آماده‌سازی ساختار دریافت و اعمال Feedback</li>
<li>حفظ Lineage و Version History</li>
<li>آماده‌سازی مسیر تولید Golden Dataset v2</li>
</ul>

<h4>Inputهای اصلی</h4>

<ul>
<li><code>Golden Dataset v1</code></li>
<li>Author Review Package مربوط به Caseهای بررسی‌شده</li>
<li>Metadata و Registryهای مرتبط در صورت وجود</li>
</ul>

<h4>Outputهای مورد انتظار</h4>

<ul>
<li>Validation ورودی‌های واقعی</li>
<li>ساختار استاندارد Feedback Integration</li>
<li>Mapping بین Review Caseها و Dataset</li>
<li>آمادگی برای تولید Golden Dataset v2 بعد از دریافت Feedback معتبر</li>
</ul>

<h4>Scope</h4>

<ul>
<li>Discovery و Validation Artifactهای موجود</li>
<li>آماده‌سازی Workflow ادغام Feedback</li>
<li>ثبت وضعیت و Lineage تغییرات</li>
</ul>

<h4>Out of Scope</h4>

<ul>
<li>تغییر مستقیم Golden Dataset v1</li>
<li>اعمال تغییر بدون تأیید Expert</li>
<li>ساخت Golden Dataset v2 قبل از دریافت Feedback</li>
<li>Overwrite کردن نسخه‌های قبلی</li>
</ul>

<h4>Final PASS Condition</h4>

<ul>
<li>ورودی‌های واقعی شناسایی و Validate شده باشند.</li>
<li>مسیر Integration آماده باشد.</li>
<li>هیچ Artifact قبلی بدون دلیل تغییر نکرده باشد.</li>
</ul>

</div>

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Starting Point / Execution Plan</h3>

<p>
این Notebook از وضعیت فعلی پروژه شروع می‌کند که در آن
<code>Golden Dataset v1</code>
تولید شده و Review Package برای بررسی Expert آماده شده است.
</p>

<p>
اولین مرحله اجرایی این Notebook، بررسی وضعیت واقعی Repository و Artifactهای موجود است تا مشخص شود چه فایل‌ها، نسخه‌ها و خروجی‌هایی قبلاً ساخته شده‌اند و Source of Truth فعلی چیست.
</p>

<h4>اولین Step اجرایی</h4>

<ul>
<li>Discovery فایل‌ها و Artifactهای مرتبط</li>
<li>بررسی وجود Golden Dataset v1</li>
<li>بررسی Review Package</li>
<li>بررسی Version و Naming Conventionهای موجود</li>
</ul>

<h4>چرا این Step لازم است؟</h4>

<p>
قبل از ساخت هر Artifact جدید باید وضعیت واقعی پروژه مشخص شود تا از دوباره‌کاری، overwrite اشتباه و ایجاد نسخه‌های ناسازگار جلوگیری شود.
</p>

<h4>Validation این مرحله</h4>

<ul>
<li>فایل‌های ورودی شناسایی شوند.</li>
<li>مسیرهای واقعی مشخص شوند.</li>
<li>Artifact تکراری یا جایگزین‌شده مشخص شود.</li>
<li>وضعیت اولیه Notebook ثبت شود.</li>
</ul>

</div>

In [5]:
# Code Cell 01

from pathlib import Path
from collections import Counter

import hashlib
import json
import os
import platform
import re
import subprocess

print("Bootstrap completed.")
print("Environment ready.")

Bootstrap completed.
Environment ready.


In [6]:
# Code Cell 02


project_root = Path.cwd()

print("Current working directory:")
print(project_root)

print("\nTop-level items:")

for item in sorted(project_root.iterdir()):
    print("-", item.name)


Current working directory:
/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2

Top-level items:
- .DS_Store
- 00_environment_and_baseline.ipynb
- 01_website_ingestion.ipynb
- 02_website_acceptance.ipynb
- 03_book_inventory.ipynb
- 04_book_ingestion.ipynb
- 05_transcript_inventory.ipynb
- 06_transcript_ingestion.ipynb
- 07_unified_schema_and_lineage.ipynb
- 08_persian_normalization.ipynb
- 09_deduplication_and_quality.ipynb
- 10_taxonomy_and_knowledge_engineering.ipynb
- 11_chunking_benchmark.ipynb
- 12_golden_dataset.ipynb
- 12b_golden_dataset_continuation_compact.ipynb
- 12c_human_review_for_author.ipynb
- 12d_second_human_review.ipynb
- 12e_golden_dataset_full_generation.ipynb
- 12f_author_feedback_integration.ipynb
- 13_embedding_benchmark.ipynb
- archive


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Step 1 — Artifact Discovery Validation</h3>

<p>
در این مرحله وضعیت واقعی Repository بررسی شد تا مشخص شود Artifactهای مورد نیاز برای ادامه مسیر وجود دارند یا خیر.
</p>

<p>
هدف این Step جلوگیری از ساخت مجدد Artifactهای موجود و مشخص کردن Source of Truth فعلی قبل از هرگونه Integration است.
</p>

<h4>Discovery انجام‌شده</h4>

<ul>
<li>مسیر فعلی Notebook مشخص شد.</li>
<li>Notebookهای مرتبط با Phase 2 و Golden Dataset شناسایی شدند.</li>
<li>Artifactهای موجود مربوط به Golden Dataset بررسی اولیه شدند.</li>
<li>وجود نسخه اولیه <code>Golden Dataset v1</code> تأیید شد.</li>
</ul>

<h4>Validation مورد انتظار</h4>

<ul>
<li>Artifactهای ورودی قابل دسترسی باشند.</li>
<li>نسخه‌های قبلی قابل شناسایی باشند.</li>
<li>قبل از ایجاد Version جدید، وضعیت Version فعلی مشخص باشد.</li>
</ul>

</div>

In [7]:
# Code Cell 03


project_root = Path.cwd()

expected_artifacts = [
    "golden_dataset_v1.jsonl",
    "golden_dataset_v1_final_report.json",
    "12e_golden_dataset_full_generation.ipynb",
    "12d_second_human_review.ipynb",
    "12c_human_review_for_author.ipynb",
]

print("Artifact validation started.\n")

missing = []

for artifact in expected_artifacts:
    path = project_root / artifact
    
    if path.exists():
        print("FOUND:", artifact)
    else:
        print("MISSING:", artifact)
        missing.append(artifact)

print("\nValidation Result:")

if not missing:
    print("Artifact discovery validation: PASS")
else:
    print("Artifact discovery validation: FAIL")
    print("Missing artifacts:", missing)


Artifact validation started.

MISSING: golden_dataset_v1.jsonl
MISSING: golden_dataset_v1_final_report.json
FOUND: 12e_golden_dataset_full_generation.ipynb
FOUND: 12d_second_human_review.ipynb
FOUND: 12c_human_review_for_author.ipynb

Validation Result:
Artifact discovery validation: FAIL
Missing artifacts: ['golden_dataset_v1.jsonl', 'golden_dataset_v1_final_report.json']


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Step 2 — Golden Dataset v1 Schema Inspection</h3>

<p>
در این مرحله ساختار واقعی <code>golden_dataset_v1.jsonl</code> بررسی می‌شود تا مشخص شود نسخه فعلی Golden Dataset چه فیلدهایی دارد و چگونه باید Feedback مؤلف به آن متصل شود.
</p>

<p>
هدف این Step جلوگیری از طراحی Feedback Integration بر اساس فرضیات است. ساختار واقعی Dataset باید قبل از ایجاد Version جدید بررسی شود.
</p>

<h4>بررسی‌های این مرحله</h4>

<ul>
<li>تعداد رکوردهای Golden Dataset v1</li>
<li>نام Fieldهای اصلی</li>
<li>نمونه یک یا چند رکورد</li>
<li>وجود شناسه پایدار برای اتصال Review</li>
<li>وجود Metadata مربوط به Lineage و Citation در صورت وجود</li>
</ul>

<h4>PASS Condition</h4>

<ul>
<li>Schema واقعی Dataset مشخص شده باشد.</li>
<li>فیلدهای مورد نیاز برای Mapping شناسایی شده باشند.</li>
<li>هیچ تصمیم طراحی بدون مشاهده Artifact واقعی گرفته نشده باشد.</li>
</ul>

</div>

In [8]:
# Code Cell 04


golden_path = Path("golden_dataset_v1.jsonl")

record_count = 0
first_record = None

with golden_path.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            record = json.loads(line)
            record_count += 1
            
            if first_record is None:
                first_record = record

print("Golden Dataset v1 inspection")
print("-----------------------------")
print("Record count:", record_count)

print("\nFirst record keys:")
for key in first_record.keys():
    print("-", key)

print("\nFirst record preview:")
print(json.dumps(first_record, ensure_ascii=False, indent=2)[:2000])


FileNotFoundError: [Errno 2] No such file or directory: 'golden_dataset_v1.jsonl'

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Step 3 — Author Review Package Discovery</h3>

<p>
در این مرحله Artifact مربوط به Review مؤلف بررسی می‌شود تا مشخص شود بازخورد Expert با چه ساختاری ذخیره شده و چگونه باید به Golden Dataset v1 متصل شود.
</p>

<p>
هدف این Step پیدا کردن Source of Truth برای Feedback است؛ بدون فرض کردن نام فایل، ساختار یا Fieldهای Review.
</p>

<h4>بررسی‌های این مرحله</h4>

<ul>
<li>پیدا کردن فایل Review Package</li>
<li>بررسی نام و نسخه Artifact</li>
<li>بررسی ساختار Feedback</li>
<li>شناسایی شناسه اتصال به Golden Dataset</li>
<li>بررسی وضعیت فعلی Review</li>
</ul>

<h4>PASS Condition</h4>

<ul>
<li>Artifact Review واقعی شناسایی شود.</li>
<li>ساختار آن قابل خواندن باشد.</li>
<li>روش اتصال Review به Dataset مشخص شود.</li>
</ul>

</div>

In [ ]:
# Code Cell 05


project_root = Path.cwd()

keywords = [
    "review",
    "author",
    "pilot",
    "feedback",
    "golden"
]

print("Searching related artifacts...\n")

matches = []

for path in project_root.rglob("*"):
    if path.is_file():
        name = path.name.lower()
        
        if any(keyword in name for keyword in keywords):
            matches.append(path)

for item in sorted(matches):
    print("-", item.relative_to(project_root))

print("\nTotal matches:", len(matches))


Searching related artifacts...

- 12_golden_dataset.ipynb
- 12b_golden_dataset_continuation_compact.ipynb
- 12c_human_review_for_author.ipynb
- 12d_second_human_review.ipynb
- 12e_golden_dataset_full_generation.ipynb
- 12f_author_feedback_integration.ipynb
- archive/author_review_22_pilots_REVISED.docx
- archive/author_review_22_pilots_REVISED.md
- archive/author_review_28_pilots.before_cell109_20260909T135135Z.md
- archive/author_review_28_pilots.before_cell111_20260909T143011Z.md
- archive/author_review_28_pilots.before_cell111_rtl_repair_20260909T154105Z.md
- archive/author_review_28_pilots.before_cell118_source_text_20260909T170159Z.md
- archive/author_review_28_pilots.md
- golden_dataset_v1.jsonl
- golden_dataset_v1_final_report.json

Total matches: 15


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Step 4 — Review Package Schema Inspection</h3>

<p>
در این مرحله محتوای واقعی Review Package بررسی می‌شود تا ساختار بازخورد Expert مشخص شود.
</p>

<p>
هدف این Step تعیین Source of Truth برای Feedback و پیدا کردن روش اتصال آن به رکوردهای Golden Dataset v1 است.
</p>

<h4>بررسی‌های این مرحله</h4>

<ul>
<li>خواندن Review Package موجود</li>
<li>بررسی تعداد Caseها</li>
<li>شناسایی ساختار هر Case</li>
<li>بررسی وجود Decision یا Feedback مؤلف</li>
<li>بررسی شناسه اتصال به Dataset</li>
</ul>

<h4>PASS Condition</h4>

<ul>
<li>ساختار Review Package مشخص شود.</li>
<li>وضعیت Feedback واقعی مشخص شود.</li>
<li>روش Mapping با Golden Dataset v1 مشخص شود.</li>
</ul>

</div>

In [ ]:
# Code Cell 06


review_path = Path("archive/author_review_28_pilots.md")

print("Review package:")
print(review_path)

print("\nFile exists:", review_path.exists())

if review_path.exists():
    text = review_path.read_text(encoding="utf-8")

    print("\nCharacter count:", len(text))

    print("\nFirst 3000 characters:")
    print(text[:3000])


Review package:
archive/author_review_28_pilots.md

File exists: True

Character count: 163318

First 3000 characters:
<div dir="rtl" align="right">

<h1 dir="rtl" align="right">بسته بازبینی مؤلف — ۲۸ مورد واقعی Golden Dataset</h1>

<p dir="rtl" align="right">این نسخه برای بازبینی دکتر کمیل رودی تهیه شده است. در هر مورد، ابتدا منبع و تاریخچه واقعی بازبینی نمایش داده می‌شود؛ اگر Target یا منبع عوض شده باشد، منبع اولیه و وضعیت سؤال/پاسخ آن قبل از منبع جایگزین آمده است.</p>

<p dir="rtl" align="right">ترتیب هر مورد ثابت است: متن منبع و تاریخچه منبع → روند سؤال و پاسخ → تصمیم‌های بازبینی → سؤال و پاسخ نهایی → نظر مؤلف.</p>

</div>


---


<div dir="rtl" align="right">

<h2 dir="rtl" align="right">مورد 1 از ۲۸</h2>

<h3 dir="rtl" align="right">۱. متن منبع (بخشی از متن اصلی)</h3>

<p dir="rtl" align="right">تلقی مردم از فیل، به عنوان ناودان و ستون و تخت و بادبزن، آنجا تبدیل به مشکل می شود که بخواهند فیل را بر اساس همان تصویر خود، به کار بگیرند. استفاده از فیل به عنوان ستون ممکن است مناسب به 

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Step 5 — Review Case Mapping Discovery</h3>

<p>
در این مرحله ساختار Caseهای داخل Review Package بررسی می‌شود تا روش اتصال هر مورد بازبینی به رکوردهای Golden Dataset v1 مشخص شود.
</p>

<p>
هدف این Step پیدا کردن شناسه یا مسیر Mapping واقعی بین Review Package و Dataset است، بدون ایجاد ارتباط فرضی.
</p>

<h4>بررسی‌های این مرحله</h4>

<ul>
<li>تعداد Caseهای Review</li>
<li>وجود شناسه‌های پایدار در متن Review</li>
<li>وجود Pilot ID یا Seed ID</li>
<li>بررسی الگوی تکرار شونده هر Case</li>
<li>شناسایی Field مناسب برای Mapping</li>
</ul>

<h4>PASS Condition</h4>

<ul>
<li>روش اتصال Review Case به Golden Dataset مشخص شود.</li>
<li>در صورت نبود شناسه مستقیم، مسیر Mapping مستند شود.</li>
</ul>

</div>

In [ ]:
# Code Cell 07


review_path = Path("archive/author_review_28_pilots.md")

text = review_path.read_text(encoding="utf-8")

case_headers = re.findall(
    r"مورد\s+\d+\s+از\s+۲۸",
    text
)

print("Detected review cases:", len(case_headers))

print("\nFirst detected headers:")
for item in case_headers[:10]:
    print("-", item)


print("\nPossible identifiers:")

patterns = [
    r"candidate_id",
    r"source_seed_id",
    r"pilot[_\s-]*\d+",
    r"GD[_\s-]*\d+"
]

for pattern in patterns:
    matches = re.findall(pattern, text, flags=re.IGNORECASE)
    print(pattern, "→", len(matches))


Detected review cases: 28

First detected headers:
- مورد 1 از ۲۸
- مورد 2 از ۲۸
- مورد 3 از ۲۸
- مورد 4 از ۲۸
- مورد 5 از ۲۸
- مورد 6 از ۲۸
- مورد 7 از ۲۸
- مورد 8 از ۲۸
- مورد 9 از ۲۸
- مورد 10 از ۲۸

Possible identifiers:
candidate_id → 0
source_seed_id → 0
pilot[_\s-]*\d+ → 0
GD[_\s-]*\d+ → 0


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Step 6 — Review Case Structure Extraction</h3>

<p>
در این مرحله ساختار داخلی Caseهای Review Package استخراج می‌شود تا مشخص شود چه اطلاعاتی برای اتصال به Golden Dataset وجود دارد.
</p>

<p>
هدف این Step شناسایی Fieldهای واقعی داخل Review Caseها است؛ مانند Source Text، Question، Answer، Decision و سایر Metadataهای موجود.
</p>

<h4>بررسی‌های این مرحله</h4>

<ul>
<li>استخراج بخش‌های تکرارشونده هر Case</li>
<li>شناسایی Headingهای ثابت</li>
<li>بررسی وجود Source Text</li>
<li>بررسی وجود سؤال نهایی</li>
<li>بررسی وجود پاسخ نهایی</li>
<li>بررسی وجود تصمیم بازبینی</li>
</ul>

<h4>PASS Condition</h4>

<ul>
<li>Template واقعی هر Review Case مشخص شود.</li>
<li>Fieldهای قابل استفاده برای Mapping مشخص شوند.</li>
</ul>

</div>

In [ ]:
# Code Cell 08


review_path = Path("archive/author_review_28_pilots.md")

text = review_path.read_text(encoding="utf-8")

# Extract markdown/html headings
headings = re.findall(
    r"<h[1-4].*?>(.*?)</h[1-4]>",
    text,
    flags=re.DOTALL
)

clean_headings = []

for h in headings:
    clean = re.sub("<.*?>", "", h)
    clean = clean.strip()
    if clean:
        clean_headings.append(clean)

counter = Counter(clean_headings)

print("Total headings detected:", len(clean_headings))

print("\nMost common headings:")

for heading, count in counter.most_common(20):
    print(f"{count}x - {heading}")


Total headings detected: 220

Most common headings:
28x - ۱. متن منبع (بخشی از متن اصلی)
28x - ۲. سؤال‌ها و پاسخ‌ها — روند اصلاح و بازبینی
28x - ۳. تصمیم من در روند بازبینی
28x - ۴. سؤال و پاسخ نهایی (مورد بازبینی توسط دکتر)
28x - ۵. نظر مؤلف (دکتر کمیل رودی)
6x - منبع اولیه
6x - سؤال و پاسخ مربوط به منبع اولیه
6x - دلیل بازبینی و تغییر منبع
6x - منبع بازبینی‌شده / جایگزین
6x - متن‌های منبع بازیابی‌شده و تأییدشده
6x - سابقه واقعی دموی فاز یک
2x - بخش‌های متکی به بازخورد مستقیم مؤلف
1x - بسته بازبینی مؤلف — ۲۸ مورد واقعی Golden Dataset
1x - مورد 1 از ۲۸
1x - مورد 2 از ۲۸
1x - مورد 3 از ۲۸
1x - مورد 4 از ۲۸
1x - مورد 5 از ۲۸
1x - مورد 6 از ۲۸
1x - مورد 7 از ۲۸


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Step 7 — Review Case Parsing Preparation</h3>

<p>
در این مرحله Caseهای Review Package به ساختار قابل پردازش تبدیل می‌شوند تا امکان مقایسه با Golden Dataset v1 فراهم شود.
</p>

<p>
هدف این Step ساخت Parser نهایی نیست؛ فقط استخراج ساختار اولیه Caseها برای کشف Mapping واقعی است.
</p>

<h4>اطلاعات استخراجی مورد نیاز</h4>

<ul>
<li>شماره Case</li>
<li>متن منبع</li>
<li>سؤال نهایی</li>
<li>پاسخ نهایی</li>
<li>تصمیم بازبینی</li>
<li>نظر مؤلف در صورت وجود</li>
</ul>

<h4>PASS Condition</h4>

<ul>
<li>۲۸ Case به‌صورت جداگانه قابل پردازش باشند.</li>
<li>Fieldهای اصلی هر Case استخراج شوند.</li>
<li>ساختار برای مقایسه با Golden Dataset آماده شود.</li>
</ul>

</div>

In [ ]:
# Code Cell 09


review_path = Path("archive/author_review_28_pilots.md")

text = review_path.read_text(encoding="utf-8")

# Remove document introduction before first case
first_case_marker = re.search(
    r"<h2.*?>مورد 1 از ۲۸</h2>",
    text
)

if first_case_marker:
    text = text[first_case_marker.start():]

cases = re.split(
    r"<h2.*?>مورد \d+ از ۲۸</h2>",
    text
)

# Remove empty sections
cases = [c.strip() for c in cases if c.strip()]

print("Extracted cases:", len(cases))

print("\nFirst case preview:")
print(cases[0][:2000])


Extracted cases: 28

First case preview:
<h3 dir="rtl" align="right">۱. متن منبع (بخشی از متن اصلی)</h3>

<p dir="rtl" align="right">تلقی مردم از فیل، به عنوان ناودان و ستون و تخت و بادبزن، آنجا تبدیل به مشکل می شود که بخواهند فیل را بر اساس همان تصویر خود، به کار بگیرند. استفاده از فیل به عنوان ستون ممکن است مناسب به نظر برسد اما فیل یک کل به‌هم‌ پیوسته است و در واقعیت، پای فیل جدای گوش و خرطوم و عاجش نیست.</p>

<h3 dir="rtl" align="right">۲. سؤال‌ها و پاسخ‌ها — روند اصلاح و بازبینی</h3>

<table dir="rtl">
<thead>
<tr>
<th align="right">مرحله</th>
<th align="right">متن سؤال</th>
<th align="right">متن پاسخ</th>
<th align="right">توضیحات / دلیل تغییر</th>
</tr>
</thead>
<tbody>
<tr>
<td align="right">نسخه اولیه سؤال</td>
<td align="right">چرا به کار گرفتن فیل بر اساس تصویری که فقط از یکی از بخش‌های آن داریم، می‌تواند مشکل‌ساز باشد؟</td>
<td align="right">—</td>
<td align="right">سؤال قبلی بیش از حد خودِ مثال فیل را موضوع سؤال قرار می‌داد. سؤال باید برای مخاطب عمومی ساده‌تر باشد و روی مف

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Step 8 — Review Case Field Extraction</h3>

<p>
در این مرحله از Caseهای استخراج‌شده، Fieldهای اصلی مورد نیاز برای Mapping با Golden Dataset v1 استخراج می‌شوند.
</p>

<p>
هدف این Step تبدیل Review Package از یک متن انسانی به یک ساختار قابل بررسی است، بدون اینکه هنوز تغییری روی Dataset اعمال شود.
</p>

<h4>Fieldهای مورد استخراج</h4>

<ul>
<li>شماره Case</li>
<li>متن منبع</li>
<li>سؤال نهایی</li>
<li>پاسخ نهایی</li>
<li>تصمیم بازبینی</li>
<li>نظر مؤلف</li>
</ul>

<h4>PASS Condition</h4>

<ul>
<li>برای Caseهای موجود ساختار داده‌ای اولیه ساخته شود.</li>
<li>Fieldهای اصلی قابل مشاهده باشند.</li>
<li>آماده مقایسه با Golden Dataset v1 باشد.</li>
</ul>

</div>

In [ ]:
# Code Cell 10



review_path = Path("archive/author_review_28_pilots.md")

text = review_path.read_text(encoding="utf-8")


# Remove introduction
first_case_marker = re.search(
    r"<h2.*?>مورد 1 از ۲۸</h2>",
    text
)

text = text[first_case_marker.start():]


cases = re.split(
    r"<h2.*?>مورد \d+ از ۲۸</h2>",
    text
)

cases = [c.strip() for c in cases if c.strip()]


def clean_html(value):
    value = re.sub(r"<.*?>", "", value)
    value = value.replace("&nbsp;", " ")
    return value.strip()


parsed_cases = []


for idx, case in enumerate(cases, start=1):

    source_match = re.search(
        r"۱\. متن منبع.*?<p.*?>(.*?)</p>",
        case,
        flags=re.DOTALL
    )

    final_section = re.search(
        r"۴\. سؤال و پاسخ نهایی.*?(?=<h3|$)",
        case,
        flags=re.DOTALL
    )

    author_section = re.search(
        r"۵\. نظر مؤلف.*?(?=<h3|$)",
        case,
        flags=re.DOTALL
    )

    decision_section = re.search(
        r"۳\. تصمیم من در روند بازبینی.*?(?=<h3|$)",
        case,
        flags=re.DOTALL
    )


    question_candidates = re.findall(
        r"(?:بازنویسی سؤال|سؤال نهایی|سؤال اصلاح‌شده).*?</td>\s*<td[^>]*>(.*?)</td>",
        case,
        flags=re.DOTALL
    )


    cleaned_questions = []

    for q in question_candidates:

        q = clean_html(q)

        if (
            q
            and q not in [
                "پذیرفته شد",
                "نیاز به اصلاح",
                "رد شد"
            ]
            and len(q) > 10
        ):
            cleaned_questions.append(q)


    # Remove duplicate questions while preserving order
    unique_questions = list(dict.fromkeys(cleaned_questions))


    parsed_cases.append({
        "case_number": idx,
        "source_text": clean_html(source_match.group(1)) if source_match else None,
        "final_question": unique_questions[-1] if unique_questions else None,
        "all_question_candidates": unique_questions,
        "final_answer_section": clean_html(final_section.group(0)) if final_section else None,
        "decision_section": clean_html(decision_section.group(0)) if decision_section else None,
        "author_section": clean_html(author_section.group(0)) if author_section else None,
    })


print("Parsed cases:", len(parsed_cases))

print("\nFirst case normalized output:")

print("Case:", parsed_cases[0]["case_number"])

print("\nFinal Question:")
print(parsed_cases[0]["final_question"])

print("\nQuestion count:")
print(len(parsed_cases[0]["all_question_candidates"]))


Parsed cases: 28

First case normalized output:
Case: 1

Final Question:
چه مشکلی پیش می‌آید اگر بر اساس شناختی که فقط از یک بخش داریم، درباره کل موضوع تصمیم بگیریم؟

Question count:
1


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Step 12 — Lineage Based Mapping Preparation</h3>

<p>
در این مرحله ارتباط بین Review Caseها و Golden Dataset v1 بر اساس اطلاعات Lineage بررسی می‌شود.
</p>

<p>
از آنجا که تطبیق مستقیم Source و Question موفق نبود، از Metadata تولیدشده در Pipeline برای پیدا کردن ارتباط استفاده می‌شود.
</p>

<h4>بررسی‌ها</h4>

<ul>
<li>بررسی مقدارهای <code>source_lineage</code></li>
<li>بررسی <code>source_seed_id</code></li>
<li>مقایسه با اطلاعات Pilot/Seed در Review Package</li>
<li>بررسی اینکه آیا Mapping از مسیر تولید اولیه قابل بازیابی است</li>
</ul>

<h4>PASS Condition</h4>

<ul>
<li>مسیر ارتباط Review Case با Golden Dataset مشخص شود.</li>
<li>در صورت نبود Mapping، دلیل آن مستند شود.</li>
</ul>

</div>

In [ ]:
# Code Cell 11

golden_path = Path("golden_dataset_v1.jsonl")

golden_records = []

with golden_path.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            golden_records.append(json.loads(line))

print("Golden records loaded:", len(golden_records))


Golden records loaded: 573


In [ ]:
# Code Cell 12



print("Unique source_lineage values:")
lineages = Counter(
    record.get("source_lineage")
    for record in golden_records
    if record.get("source_lineage")
)

print("Count:", len(lineages))

print("\nFirst 30 lineage values:")

for lineage, count in lineages.most_common(30):
    print(count, "->", lineage)


print("\n\nSource seed examples:")

seeds = [
    record.get("source_seed_id")
    for record in golden_records
    if record.get("source_seed_id") is not None
]

print("Total seeds:", len(set(seeds)))
print("First 30 seeds:", sorted(set(seeds))[:30])


Unique source_lineage values:
Count: 28

First 30 lineage values:
21 -> pilot_21
21 -> pilot_1
21 -> pilot_12
21 -> pilot_2
21 -> pilot_3
21 -> pilot_4
21 -> pilot_5
21 -> pilot_6
21 -> pilot_7
21 -> pilot_8
21 -> pilot_9
21 -> pilot_10
21 -> pilot_11
20 -> pilot_13
20 -> pilot_14
20 -> pilot_15
20 -> pilot_16
20 -> pilot_17
20 -> pilot_18
20 -> pilot_19
20 -> pilot_20
20 -> pilot_22
20 -> pilot_23
20 -> pilot_24
20 -> pilot_25
20 -> pilot_26
20 -> pilot_27
20 -> pilot_28


Source seed examples:
Total seeds: 28
First 30 seeds: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28]


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Step 18 — Golden Dataset v1 Report Inspection</h3>

<p>
در این مرحله گزارش نهایی تولید Golden Dataset v1 بررسی می‌شود تا مشخص شود منطق انتخاب، تعداد رکوردهای نهایی و وضعیت Version قبلی چگونه بوده است.
</p>

<p>
هدف این Step جلوگیری از تعریف اشتباه Golden Dataset v2 و حفظ Lineage نسخه قبلی است.
</p>

<h4>بررسی‌ها</h4>

<ul>
<li>ساختار گزارش نهایی v1</li>
<li>تعداد رکوردهای پذیرفته‌شده</li>
<li>قوانین انتخاب Candidate</li>
<li>Metadata مربوط به Version</li>
</ul>

<h4>PASS Condition</h4>

<ul>
<li>جایگاه Golden Dataset v1 مشخص شود.</li>
<li>قانون ساخت v2 بر اساس Artifactهای واقعی تعیین شود.</li>
</ul>

</div>

In [ ]:
# Code Cell 13


report_path = Path("golden_dataset_v1_final_report.json")

print("Report exists:", report_path.exists())

with report_path.open("r", encoding="utf-8") as f:
    report = json.load(f)

print("\nTop-level keys:")
for key in report.keys():
    print("-", key)

print("\nReport preview:")
print(json.dumps(report, ensure_ascii=False, indent=2)[:3000])


Report exists: True

Top-level keys:
- dataset_version
- freeze_status
- created_at
- target_size
- records
- artifact
- coverage
- validation
- status

Report preview:
{
  "dataset_version": "golden_dataset_v1",
  "freeze_status": "PASS",
  "created_at": "2026-09-10T14:12:45.079766",
  "target_size": 573,
  "records": 573,
  "artifact": {
    "path": "golden_dataset_v1.jsonl",
    "sha256": "25ed3732db5f03d3b771daa7b1051d61f8de7524644c4a055792b39a22493cc4"
  },
  "coverage": {
    "seeds": 28,
    "topics": 7,
    "competencies": 24
  },
  "validation": {
    "duplicate_ids": 0,
    "duplicate_questions": 0,
    "missing_answers": 0,
    "missing_evidence": 0
  },
  "status": "READY_FOR_NEXT_EVALUATION_PHASE"
}


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Step 19 — NB12e Generation Lineage Inspection</h3>

<p>
گزارش نهایی Golden Dataset v1 تأیید کرد که نسخه اول با ۵۷۳ رکورد Freeze شده و هر ۲۸ Seed در آن پوشش داده شده‌اند.
بنابراین پیش از طراحی Golden Dataset v2 باید منطق واقعی تولید این ۵۷۳ رکورد از ۲۸ Pilot در Notebook قبلی بررسی شود.
</p>

<p>
در این مرحله Notebook
<code>12e_golden_dataset_full_generation.ipynb</code>
به‌صورت Read-Only بررسی می‌شود تا مشخص شود
<code>source_seed_id</code>،
<code>source_lineage</code>
و Candidateهای تولیدشده چگونه ساخته و در Artifact نهایی پذیرفته شده‌اند.
</p>

<h4>بررسی‌های این مرحله</h4>

<ul>
<li>پیدا کردن Cellهای مرتبط با <code>source_seed_id</code> و <code>source_lineage</code></li>
<li>پیدا کردن منطق ساخت <code>golden_dataset_v1.jsonl</code></li>
<li>بررسی اینکه آیا Dataset نهایی از همه Candidateهای معتبر ساخته شده است یا Selection دیگری وجود دارد</li>
<li>شناسایی نقطه‌ای که Feedback مؤلف باید در Lineage وارد شود</li>
</ul>

<h4>PASS Condition</h4>

<ul>
<li>منطق واقعی Seed → Generated Records → Golden Dataset v1 از خود NB12e قابل بازیابی باشد.</li>
<li>بدون حدس مشخص شود تغییر یک Pilot چه اثری بر نسخه بعدی Dataset خواهد داشت.</li>
</ul>

</div>

In [ ]:
# Code Cell 14

nb12e_path = Path("12e_golden_dataset_full_generation.ipynb")

with nb12e_path.open("r", encoding="utf-8") as f:
    nb12e = json.load(f)

search_terms = [
    "source_seed_id",
    "source_lineage",
    "golden_dataset_v1",
    "generation_status",
    "target_size",
]

matches = []

for cell_index, cell in enumerate(nb12e.get("cells", [])):
    source = "".join(cell.get("source", []))

    found_terms = [
        term for term in search_terms
        if term in source
    ]

    if found_terms:
        matches.append({
            "cell_index": cell_index,
            "cell_type": cell.get("cell_type"),
            "terms": found_terms,
            "source": source,
        })

print("Relevant NB12e cells found:", len(matches))

for item in matches:
    print("\n" + "=" * 90)
    print("Notebook cell index:", item["cell_index"])
    print("Cell type:", item["cell_type"])
    print("Matched terms:", ", ".join(item["terms"]))
    print("-" * 90)
    print(item["source"][:2500])


Relevant NB12e cells found: 41

Notebook cell index: 43
Cell type: code
Matched terms: target_size
------------------------------------------------------------------------------------------
# Code Cell 21

# --------------------------------------------------
# Expansion Planning
# --------------------------------------------------

target_dataset_size = 500

seed_count = len(final_seed_registry)


# Base allocation:
# each seed keeps representation,
# then extra capacity goes to coverage gaps.

base_per_seed = 5

base_samples = seed_count * base_per_seed

remaining_capacity = (
    target_dataset_size
    -
    base_samples
)


gap_competencies = [
    item["competency_id"]
    for item in competency_gap_report
    if item["status"] == "GAP"
]


low_coverage_competencies = [
    item["competency_id"]
    for item in expansion_priorities
    if item["coverage_count"] <= 1
]


expansion_plan = {
    "target_size": target_dataset_size,
    "seed_count": seed_count,
    "base_samples_per_s

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Final State — Waiting for Author Feedback</h3>

<p>
این Notebook تا مرحله آماده‌سازی Integration بازخورد مؤلف پیش رفته است.
</p>

<ul>
<li>Golden Dataset v1 با موفقیت بررسی و Freeze شده است.</li>
<li>Review Package شامل ۲۸ مورد بازبینی برای ارسال به مؤلف آماده شده است.</li>
<li>Mapping بین Review Caseها و Pilot/Seedهای اصلی مشخص شده است.</li>
<li>هیچ تغییری روی Golden Dataset v1 اعمال نشده است.</li>
</ul>

<p>
در این مرحله فرآیند متوقف می‌شود و منتظر دریافت نظر نهایی مؤلف می‌مانیم.
پس از برگشت فایل بازبینی‌شده، همین Notebook ادامه داده می‌شود تا:
</p>

<ul>
<li>Feedback مؤلف استخراج شود.</li>
<li>تغییرات تأییدشده اعمال شوند.</li>
<li>Golden Dataset v2 ساخته و Validation شود.</li>
</ul>

<h4>Current Status</h4>

<p>
WAITING_FOR_AUTHOR_FEEDBACK
</p>

</div>


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## یادداشت ادامهٔ کار پس از دریافت بازخورد مؤلف: بازبینی کیفیت Golden Dataset

Golden Dataset فعلی از نظر ساختار، شمار رکوردها، شواهد، lineage و integrity معتبر است و نباید تغییر داده شود.

با این حال، Benchmark بازیابی در Notebook 13 نشان داد که بخشی از پرسش‌ها برای ارزیابی Retrieval به اندازهٔ کافی دقیق و متمایزکننده نیستند. در بعضی نمونه‌ها، پرسش موضوع اصلی منبع را کامل منتقل نمی‌کند یا بیش از حد عمومی/قالبی است. در نتیجه ممکن است Retriever نتواند منبع صحیح را بازیابی کند؛ حتی اگر Chunking، Citation و شواهد منبع سالم باشند.

این یافته به معنی نامعتبر بودن کل Golden Dataset نیست. هدف مرحلهٔ بعدی، ایجاد Golden Dataset نهایی و یکدست‌تر بر اساس بازخورد مؤلف و این گزارش تشخیصی است.

### اصول غیرقابل تغییر

- Golden Dataset فعلی دست‌نخورده باقی می‌ماند.
- هیچ رکوردی بدون ثبت lineage، دلیل تغییر و امکان مقایسه با نسخهٔ پیشین حذف یا جایگزین نمی‌شود.
- نسخهٔ جدید فقط پس از دریافت و ثبت بازخورد مؤلف ساخته می‌شود.
- Notebookهای بعدی نباید به شمارهٔ نسخه وابسته باشند؛ همگی فقط از متغیر اصلی `golden_dataset` استفاده می‌کنند.
- پس از ایجاد Dataset نهایی، فقط Notebookهای 13 به بعد دوباره اجرا می‌شوند؛ بازنویسی Notebookهای بعدی لازم نیست.

### مسیر اجرایی در این Notebook

1. بازخورد مؤلف برای هر مورد دریافت و ساختاریافته ثبت شود.
2. هر پرسش Golden با شواهد منبع، پاسخ مورد انتظار و نظر مؤلف بازبینی شود.
3. پرسش‌هایی که یکی از مشکلات زیر را دارند علامت‌گذاری شوند:

   - موضوع یا مفهوم اصلی منبع را منتقل نمی‌کنند؛
   - بیش از حد عمومی، مبهم یا قالبی هستند؛
   - یک واژه یا عبارت ناقص/نامرتبط را موضوع پرسش قرار داده‌اند؛
   - با پاسخ مورد انتظار هم‌راستا نیستند؛
   - با شواهد منبع قابل پاسخ‌گویی دقیق نیستند؛
   - از نظر سطح آموزشی، زبان یا دامنه با هدف Golden Dataset سازگار نیستند.

4. برای هر مورد علامت‌گذاری‌شده، یکی از تصمیم‌های زیر ثبت شود:

   - `KEEP`: بدون تغییر باقی بماند؛
   - `REWRITE_QUESTION`: فقط پرسش بازنویسی شود؛
   - `REWRITE_ANSWER`: فقط پاسخ بازنویسی شود؛
   - `REWRITE_BOTH`: پرسش و پاسخ بازنویسی شوند؛
   - `REPLACE_EVIDENCE`: شواهد منبع اصلاح یا جایگزین شوند؛
   - `REMOVE_WITH_REASON`: فقط با دلیل صریح و ثبت lineage حذف شود.

5. پیش از ساخت نسخهٔ نهایی، برای هر رکورد تأیید شود که:

   - پرسش دارای موضوع روشن و متمایزکننده است؛
   - پاسخ مستقیماً از شواهد ثبت‌شده پشتیبانی می‌شود؛
   - citation و attribution با منبع هم‌خوان هستند؛
   - شناسهٔ رکورد، seed و lineage قابل ردیابی باقی مانده‌اند؛
   - سؤال تکراری یا نزدیک‌به‌تکراری ناخواسته ایجاد نشده است.

6. Golden Dataset نهایی ساخته و freeze شود؛ سپس گزارش کامل تغییرات شامل موارد نگه‌داشته‌شده، بازنویسی‌شده، جایگزین‌شده و حذف‌شده تولید شود.

7. نسخهٔ نهایی در محل استاندارد Golden Dataset قرار گیرد. سل آغازین Notebookهای بعدی به‌صورت خودکار همان Dataset فعال را انتخاب می‌کند.

8. پس از آن، Notebookهای 13 به بعد دوباره اجرا شوند تا Benchmarkهای Retrieval، Citation، Attribution، Context و تصمیم نهایی با Dataset کامل و نهایی محاسبه شوند.

### معیار موفقیت

Golden Dataset نهایی باید هم از نظر صحت منبع و هم از نظر کیفیت پرسش، برای ارزیابی منصفانهٔ Retrieval قابل اتکا باشد؛ به‌گونه‌ای که شکست بازیابی تا حد ممکن ناشی از ضعف واقعی سیستم باشد، نه ابهام یا نقص در صورت‌بندی پرسش.

</div>